# Air Quality Gold

Executable SQL transformations and orchestration live in this notebook. Only runtime checks are imported from `src.validation.gold`. Execute weather → air quality → earthquakes. All persistent writes are restricted to the explicitly approved catalog's gold schema — `weather_api.gold` (production) or `weather_api_test.gold` (test). No job or dashboard is created.

## Parameters
`project_root` must contain this checkout and the notebook JSON sources. The catalog/schema parameters are validated against the explicitly approved target (`weather_api`/`gold` production, `weather_api_test`/`gold` test); a blank catalog fails validation instead of silently defaulting.

In [0]:
# Minimal source locator; all validation/loading rules live in Weather bootstrap.
from pathlib import Path
import json
import sys
dbutils.widgets.text('project_root','')
_override=dbutils.widgets.get('project_root')
if _override and not Path(_override).expanduser().is_absolute():raise ValueError('project_root must be absolute, or blank for discovery')
_candidates=[Path(_override).expanduser().resolve()] if _override else [Path.cwd().resolve(),*Path.cwd().resolve().parents]
_root=next((p for p in _candidates if (p/'src/config.py').is_file() and (p/'notebooks/gold/weather_gold.ipynb').is_file()),None)
if _root is None:raise FileNotFoundError('Provide absolute project_root with src and notebook JSON sources; Git-folder source export may be required')
_boot=json.loads((_root/'notebooks/gold/weather_gold.ipynb').read_text())
_boot_cells=[c for c in _boot['cells'] if c.get('cell_type')=='code' and c.get('metadata',{}).get('tags')==['gold_bootstrap']]
if len(_boot_cells)!=1:raise ValueError('Expected exactly one canonical gold_bootstrap cell')
exec(compile(''.join(_boot_cells[0]['source']),'weather_gold bootstrap','exec'),globals())
project_root=resolve_project_root(str(_root))
sys.path.insert(0,str(project_root))
dbutils.widgets.text('config_path','config/isolated.example.json')
dbutils.widgets.dropdown('validate_only','true',['true','false'])
config_path=resolve_config_path(project_root,dbutils.widgets.get('config_path'))
dbutils.widgets.text("release_id", "")

dbutils.widgets.text('gold_schema','gold')
dbutils.widgets.text('catalog','')  # required explicitly; approved: weather_api (production) or weather_api_test (test)


## Shared definitions
Loading definition cells has no database side effects. Successful persisted predecessors are checked again by this notebook.

In [0]:
load_gold_definitions(project_root,['weather'],globals())


## Source selection and business transformations
All joins use full UTC timestamps and release identity. Source-specific expressions are owned here.

In [0]:
# gold_definitions: AIR-QUALITY-owned transformations and combined products
US_BANDS=((50,'Good'),(100,'Moderate'),(150,'Unhealthy for sensitive groups'),(200,'Unhealthy'),(300,'Very unhealthy'))
EU_BANDS=((20,'Good'),(40,'Fair'),(60,'Moderate'),(80,'Poor'),(100,'Very poor'))

def aqi_case(column,standard):
    bands=US_BANDS if standard=='us' else EU_BANDS
    last='Hazardous' if standard=='us' else 'Extremely poor'
    return f"CASE WHEN {column} IS NULL THEN 'Unknown' WHEN {column}<0 OR isnan(cast({column} AS DOUBLE)) THEN 'Invalid' "+' '.join(f'WHEN {column}<={n} THEN {literal(label)}' for n,label in bands)+f' ELSE {literal(last)} END'

def air_quality_query(relation,registry,kind):
    return f'''SELECT b.*,{aqi_case('us_aqi','us')} AS us_aqi_category,{aqi_case('european_aqi','eu')} AS european_aqi_category,
      (pm2_5 IS NULL OR pm10 IS NULL OR us_aqi IS NULL) AS missing_core_measurement
      FROM ({measurement_base(relation,registry,'air_quality',kind)}) b'''

AQ_KEYS={'air_quality_current_latest':['location_id'],'air_quality_hourly_latest':['location_id','forecast_for'],
         'weather_aq_coverage':['source','identity_key']}
AQ_VIEW_KEYS={'environment_hourly':['location_id','forecast_for'],'location_current':['location_id'],
 'sampled_location_hourly_summary':['forecast_for'],'weather_aq_freshness':['source','identity_key']}

def weather_aq_coverage(ctx):
    parts=[];s=ctx['stages'];registry=s['dim_location']
    for source in ('weather','air_quality'):
        c=s[source+'_current_latest'];h=s[source+'_hourly_latest']
        parts.append(f'''SELECT d.location_id,cast(d.location_id AS STRING) identity_key,{literal(source)} source,
          d.location_name,d.latitude,d.longitude,d.is_active AND d.{source+'_enabled'} enabled,
          c.snapshot_at current_retrieval_at,c.observed_at current_valid_at,
          greatest(c.snapshot_at,h.retrieval) latest_retrieval_at,greatest(c.observed_at,h.last_hour) latest_valid_at,
          h.first_hour coverage_start_at,h.last_hour coverage_end_at,coalesce(h.n,0) retained_rows,
          coalesce(h.known_temperature,0) core_measurement_rows,coalesce(h.missing,0) missing_core_rows,
          c.missing_core_measurement current_missing_core,cast(c.location_id IS NOT NULL OR coalesce(h.n,0)>0 AS INT) represented_locations,
          cast(NULL AS TIMESTAMP) feed_generated_at
          FROM {registry} d LEFT JOIN {c} c ON d.location_id=c.location_id
          LEFT JOIN (SELECT location_id,count(*) n,count_if(NOT missing_core_measurement) known_temperature,
            count_if(missing_core_measurement) missing,min(forecast_for) first_hour,max(forecast_for) last_hour,max(snapshot_at) retrieval
            FROM {h} GROUP BY location_id) h ON d.location_id=h.location_id''')
    return ' UNION ALL '.join(parts)

def freshness_query(coverage,settings):
    return f'''SELECT c.*,r.published_at gold_published_at,
      timestampdiff(SECOND,c.latest_retrieval_at,current_timestamp()) retrieval_age_seconds,
      timestampdiff(SECOND,c._built_at,current_timestamp()) gold_build_age_seconds,
      cast(get_json_object(r.settings,'$.stale_seconds') AS DOUBLE) stale_after_seconds,
      CASE WHEN c.latest_retrieval_at IS NULL THEN 'No data'
        WHEN timestampdiff(SECOND,c.latest_retrieval_at,current_timestamp())>cast(get_json_object(r.settings,'$.stale_seconds') AS DOUBLE) THEN 'Stale'
        ELSE 'Within configured retrieval age' END retrieval_freshness
      FROM {coverage} c JOIN {settings} r ON c._release_id=r.release_id'''

def air_quality_views(mapping):
    w=mapping['weather_hourly_latest'];a=mapping['air_quality_hourly_latest'];d=mapping['dim_location']
    wc=mapping['weather_current_latest'];ac=mapping['air_quality_current_latest'];r=mapping['_gold_settings']
    wcols=[n for n in output_types('weather','hourly') if n not in ('location_id','forecast_for')]
    acols=[n for n in output_types('air_quality','hourly') if n not in ('location_id','forecast_for')]
    env=f'''SELECT coalesce(w._release_id,a._release_id) _release_id,
      coalesce(w.location_id,a.location_id) location_id,coalesce(w.forecast_for,a.forecast_for) forecast_for,
      coalesce(w.location_timezone,a.location_timezone) location_timezone,
      coalesce(w.valid_local_display,a.valid_local_display) valid_local_display,
      coalesce(w.local_utc_offset_seconds,a.local_utc_offset_seconds) local_utc_offset_seconds,
      w.location_id IS NOT NULL weather_available,a.location_id IS NOT NULL air_quality_available,
      timestampdiff(MICROSECOND,w.snapshot_at,a.snapshot_at)/1000000D aq_minus_weather_retrieval_seconds,
      CASE WHEN w.location_id IS NULL OR a.location_id IS NULL THEN 'single_source'
        WHEN abs(timestampdiff(MICROSECOND,w.snapshot_at,a.snapshot_at))/1000000D<=cast(get_json_object(r.settings,'$.alignment_seconds') AS DOUBLE) THEN 'within_threshold'
        ELSE 'retrievals_apart' END alignment_quality,
      {','.join('w.`'+n+'` AS weather_'+n for n in wcols)},
      {','.join('a.`'+n+'` AS air_quality_'+n for n in acols)},
      a.us_aqi_category,a.european_aqi_category,w.weather_description,
      w.missing_core_measurement weather_missing_core,a.missing_core_measurement air_quality_missing_core
      FROM {w} w FULL OUTER JOIN {a} a ON w._release_id=a._release_id AND w.location_id=a.location_id AND w.forecast_for=a.forecast_for
      JOIN {r} r ON r.release_id=coalesce(w._release_id,a._release_id)'''
    ccols=[n for n in output_types('weather','current') if n!='location_id']
    qcols=[n for n in output_types('air_quality','current') if n!='location_id']
    location=f'''SELECT d.*,w.location_id IS NOT NULL weather_available,a.location_id IS NOT NULL air_quality_available,
      {','.join('w.`'+n+'` AS weather_'+n for n in ccols)},
      {','.join('a.`'+n+'` AS air_quality_'+n for n in qcols)},
      a.us_aqi_category,a.european_aqi_category,w.weather_description,
      w.missing_core_measurement weather_missing_core,a.missing_core_measurement air_quality_missing_core
      FROM {d} d LEFT JOIN {wc} w ON d._release_id=w._release_id AND d.location_id=w.location_id
      LEFT JOIN {ac} a ON d._release_id=a._release_id AND d.location_id=a.location_id'''
    # No authoritative state boundary/membership evidence is available. This is an
    # all-sampled-locations summary, deliberately NOT a claimed statewide product.
    sampled=f'''SELECT _release_id,forecast_for,'sampled_locations_only' geographic_scope,
      count(DISTINCT location_id) represented_locations,count(weather_temperature_2m) temperature_locations,
      min(weather_temperature_2m) min_temperature_c,max(weather_temperature_2m) max_temperature_c,cast(avg(cast(weather_temperature_2m AS DECIMAL(30,12))) AS DOUBLE) mean_temperature_c,
      count(air_quality_us_aqi) us_aqi_locations,max(air_quality_us_aqi) maximum_hourly_source_us_aqi,
      {aqi_case('max(air_quality_us_aqi)','us')} maximum_hourly_source_us_aqi_category,
      min(weather_snapshot_at) weather_first_retrieval,max(weather_snapshot_at) weather_last_retrieval,
      min(air_quality_snapshot_at) aq_first_retrieval,max(air_quality_snapshot_at) aq_last_retrieval
      FROM ({env}) e GROUP BY _release_id,forecast_for'''
    return {'environment_hourly':env,'location_current':location,'sampled_location_hourly_summary':sampled,
            'weather_aq_freshness':freshness_query(mapping['weather_aq_coverage'],r)}



## Validation, publication and persistence reconciliation
All planned stage outputs are checked before stage copies are written. Failures propagate and leave the preceding active release visible.

In [0]:
def run_air_quality(session,*,config_path,release_id,project_root='.',validate_only=True,gold=None):
    ctx=prepare_context(session,config_path,gold,release_id=release_id,project_root=project_root,write_stage=None if validate_only else 'air_quality')
    try:
        checks.validate_stage_dependency(completed_stages(ctx),'air_quality')
        persisted_dependencies(ctx,WEATHER_KEYS)
        for name,q in weather_transformations(ctx).items():
            expected=ctx.temp('weather_dependency_'+name,f'SELECT {literal(ctx["release_id"])} _release_id,cast({literal(ctx["built_at"])} AS TIMESTAMP) _built_at,s.* FROM ({q}) s')
            checks.reconcile(ctx['db'],expected,ctx['stages'][name])
        queries={'air_quality_'+kind+'_latest':air_quality_query(ctx['sources']['air_quality',kind],ctx['registry'],kind) for kind in ('current','hourly')}
        counts=stage_products(ctx,queries,AQ_KEYS)
        for kind in ('current','hourly'):
            name='air_quality_'+kind+'_latest'
            checks.validate_latest(ctx['db'],ctx['sources']['air_quality',kind],ctx['stages'][name],AQ_KEYS[name])
            checks.validate_aqi_categories(ctx['db'],ctx['stages'][name])
        counts.update(stage_products(ctx,{'weather_aq_coverage':weather_aq_coverage(ctx)},AQ_KEYS))
        candidate=candidate_mapping(ctx);derived,extra=build_derived(ctx,air_quality_views(candidate),AQ_VIEW_KEYS);counts.update(extra)
        if not validate_only:
            preflight_query_family(ctx,air_quality_views(public_mapping(ctx))|{n+'_history':q for n,q in air_quality_views(history_mapping(ctx)).items()})
            active=_active_id(ctx['db'],ctx['cfg'])==release_id
            if not active:stage_status(ctx,'air_quality','BUILDING')
            try:
                preflight_and_write(ctx,AQ_KEYS,AQ_KEYS)
                install_views(ctx,air_quality_views(public_mapping(ctx)),AQ_VIEW_KEYS)
                install_history_views(ctx,air_quality_views(history_mapping(ctx)),AQ_VIEW_KEYS)
                if not active:stage_status(ctx,'air_quality','VALIDATED',counts)
            except Exception as exc:
                if not active:stage_status(ctx,'air_quality','FAILED',error=type(exc).__name__)
                raise
        return finish_result(ctx,{'stage':'air_quality','release_id':release_id,'validate_only':validate_only,'counts':counts,**ctx['report']})
    except Exception as exc:
        record_failure(ctx,exc);raise
    finally:ctx.close()


## Execute
Set `validate_only=false` deliberately to persist this stage. Pass the weather stage release_id unchanged to both later notebooks.

In [0]:
gold_result = run_air_quality(
    spark, gold=GoldConfig(catalog=dbutils.widgets.get('catalog'), schema=dbutils.widgets.get('gold_schema')), config_path=config_path, project_root=project_root,
    release_id=dbutils.widgets.get("release_id"),
    validate_only=dbutils.widgets.get("validate_only") == "true",
)


## Summary
A validated stage is not an active complete release until all three notebooks finish. Inspect `publication_status` after final publication.

In [0]:
print(json.dumps(gold_result, default=str, sort_keys=True))
